Загрузил данные в langfuse, прогнал эксперименты разными. Автоматизированный вариант из документации ниже не привязывает запуски к промпту и почему-то он создал два запуска экспериментов. Так же я нашёл вариант как привязать к промпту оценку на данных, но это работает не так как я хотел. У промпта свои единые скоры для каждой версии. Нельзя, например, увидеть оценки промптов для каждого запуска и для разных наборов данных. Но это можно увидеть в самих датасетах.

https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk

In [1]:
import os

from datasets import load_from_disk
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
from tqdm import tqdm

from utils import ActInfo, structured_model

os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-9058c552-1e8e-41c2-bcea-8abae84494da"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-1434620a-ab84-4592-99f6-e3dd7419dd64"
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

In [2]:
client = get_client()
handler = CallbackHandler()

### Create datasets and load items

In [4]:
dataset = load_from_disk("../data/raw")

In [ ]:
client.create_dataset(name="RusLawOD/train", expected_output_schema=ActInfo.model_json_schema())
client.create_dataset(name="RusLawOD/test", expected_output_schema=ActInfo.model_json_schema())

Dataset(id='cmlg7kihv000dph076dcy60yl', name='RusLawOD/test', description=None, metadata=None, input_schema=None, expected_output_schema={'properties': {'full_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Полное наименование правового акта (тип + наименование государственного органа)', 'title': 'Full Name'}, 'publication_date': {'anyOf': [{'format': 'date', 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Дата опубликования документа', 'title': 'Publication Date'}, 'number': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Номер документа (например, 3789-р)', 'title': 'Number'}, 'title': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Заголовок документа', 'title': 'Title'}, 'government_agency_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Наименование государственного органа', 'title': 'Government Agency Name'

In [12]:
for item in dataset["train"]:
    text = item["text"]

    full_name = item["full_name"]
    publication_date = item["publication_date"]
    number = item["number"]
    title = item["title"]
    government_agency_name = item["government_agency_name"]
    signatory = item["signatory"]
    type_ = item["type"]

    client.create_dataset_item(
        dataset_name="RusLawOD/train",
        input=text,
        expected_output={
            "full_name": full_name,
            "publication_date": publication_date,
            "number": number,
            "title": title,
            "government_agency_name": government_agency_name,
            "signatory": signatory,
            "type": type_,
        },
    )

for item in dataset["test"]:
    text = item["text"]

    full_name = item["full_name"]
    publication_date = item["publication_date"]
    number = item["number"]
    title = item["title"]
    government_agency_name = item["government_agency_name"]
    signatory = item["signatory"]
    type_ = item["type"]

    client.create_dataset_item(
        dataset_name="RusLawOD/test",
        input=text,
        expected_output={
            "full_name": full_name,
            "publication_date": publication_date,
            "number": number,
            "title": title,
            "government_agency_name": government_agency_name,
            "signatory": signatory,
            "type": type_,
        },
    )

### Evaluate datasets with qwen3-30b

In [3]:
text_prompt = """
Извлеки данные из документа согласно схеме

Документ:
{{document}}
"""

In [4]:
prompt_client = client.create_prompt(name="structured-output-prompt", prompt=text_prompt, config={"json_schema": ActInfo.model_json_schema()})

In [5]:
def accuracy(input, output, expected_output, **kwargs):
    scores = []
    for k, v in expected_output.items():
        if output[k] == v:
            scores.append(1)
        else:
            scores.append(0)
    return sum(scores) / len(scores)

In [6]:
test_dataset = client.get_dataset("RusLawOD/test")


In [ ]:
for item in test_dataset.items:
    with item.run(run_name="full-run-prompt-v1") as span:
        input_prompt = prompt_client.compile(document=item.input)
        output = structured_model.with_structured_output(ActInfo).invoke(input=input_prompt, config={"callbacks": [handler]})
        span.update_trace(input=input_prompt, output=output)
        span.score_trace(name="correctness", value=accuracy(item.input, output.model_dump(), item.expected_output))
    break
client.flush()

In [ ]:
# эта версия привязывает промпт к запуску
for item in tqdm(test_dataset.items):
    with client.start_as_current_observation(name="structured-gen-qwen3-30b", as_type="generation", prompt=prompt_client) as gen:
        with item.run(run_name="full-test-eval-qwen3-30b") as span:
            input_prompt = prompt_client.compile(document=item.input)
            output = structured_model.with_structured_output(ActInfo).invoke(input=input_prompt, config={"callbacks": [handler]})
            span.update_trace(input=input_prompt, output=output)
            span.score_trace(name="correctness", value=accuracy(item.input, output.model_dump(), item.expected_output))
client.flush()

 10%|█         | 10/100 [04:43<42:29, 28.33s/it]


KeyboardInterrupt: 

Рекомендуемый способ из документации

In [9]:
def accuracy(*, input, output, expected_output, metadata, **kwargs):
    scores = []
    for k, v in expected_output.items():
        if output[k] == v:
            scores.append(1)
        else:
            scores.append(0)
    avg =  sum(scores) / len(scores)
    return Evaluation(name="accuracy", value=avg)

def task(*, item, **kwargs):
    input_prompt = prompt_client.compile(document=item.input)
    output = structured_model.with_structured_output(ActInfo).invoke(input=input_prompt, config={"callbacks": [handler]})
    return output.model_dump()

In [ ]:
result = test_dataset.run_experiment(
    name="automated_test",
    task=task,
    evaluators=[accuracy]
)

Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed

Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Evaluator accuracy failed: 'ActInfo' object is not subscriptable
Item 39 failed: Error code: 400 - {'error': {'code': 'BAD_REQUEST', 'message': 'Некорректный запрос.', 'metadata': {'raw': '{"error":{"message":"This endpoint\'s maximum context length is 262144 tokens. However, you requested about 312135 tokens (312135 of text input). Please reduce the length of either one, or use the \\"middle-out\\" transform to